In [ ]:
import numpy as np
import scipy.io as sio
import time
import pandas as pd
import matplotlib.pyplot as plt
from complex_localpca import mppca
from skimage.metrics import structural_similarity as ssim

# ----------------------- Parameters -----------------------
PATCH_RADIUS = 3        # patch radius for 7×7×7
EPSILON      = 1e-12    # to avoid divide-by-zero in SNR
SLICE_IDX    = None      # None이면 중앙 슬라이스 (Z//2) 사용

# ----------------------- Load data -----------------------
# Original complex data + brain mask
orig_mat    = sio.loadmat('meas_gre_dir1.mat')
meas_gre    = orig_mat['meas_gre']                # shape (X,Y,Z,Ne), complex
mask_brain  = orig_mat['mask_brain'].astype(bool) # shape (X,Y,Z)

# Noisy real & imag (noise case)
noise_mat   = sio.loadmat('noisy_meas_gre_dir1_30.mat')
noisy_real  = noise_mat['noisy_real'].astype(np.float32)
noisy_imag  = noise_mat['noisy_imag'].astype(np.float32)

# Dimensions
X, Y, Z, Ne = meas_gre.shape
print(f"Data dimensions: {X}×{Y}×{Z}×{Ne}")
print(f"Total number of slice-echo combinations: {Z * Ne}")

X, Y, Z, Ne = meas_gre.shape
if SLICE_IDX is None:
    SLICE_IDX = Z // 2

mag_orig  = np.abs(meas_gre)
mag_noisy = np.sqrt(noisy_real**2 + noisy_imag**2)

# ----------------------- Magnitude MP-PCA -----------------------
print("▶ Running Magnitude MP-PCA...")
t1 = time.time()
den_mag = mppca(
    mag_noisy,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Magnitude MP-PCA completed in {time.time()-t1:.1f}s")
# den_mag shape: (X, Y, Z, Ne)
deno_mag = den_mag

# ----------------------- Split MP-PCA (Real/Imag) -----------------------
print("▶ Running Split MP-PCA (Real/Imag)...")
t1 = time.time()

den_real = mppca(
    noisy_real,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)

den_imag = mppca(
    noisy_imag,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Split MP-PCA completed in {time.time()-t1:.1f}s")

# den_real, den_imag shape: (X, Y, Z, Ne)
deno_split = np.sqrt(den_real**2 + den_imag**2)

# ----------------------- Complex MP-PCA -----------------------
# build multi-channel array: real & imag concatenated
multi_ch = np.concatenate([noisy_real, noisy_imag], axis=3)  # (X,Y,Z,2*Ne)

print("▶ Running Complex MP-PCA...")
t1 = time.time()
den_all = mppca(
    multi_ch,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Complex MP-PCA completed in {time.time()-t1:.1f}s")

real_den_cmplx = den_all[..., :Ne]
imag_den_cmplx = den_all[..., Ne:]
deno_cmplx = np.abs(real_den_cmplx + 1j * imag_den_cmplx)  # (X,Y,Z,Ne)

# ----------------------- Save results for future use -----------------------
print("▶ Saving denoising results...")

# Save all results in one MAT file
all_results = {
    'mag_orig': mag_orig.astype(np.float32),
    'mag_noisy': mag_noisy.astype(np.float32),
    'deno_mag': deno_mag.astype(np.float32),
    'deno_split': deno_split.astype(np.float32),
    'deno_cmplx': deno_cmplx.astype(np.float32),
    'mask_brain': mask_brain.astype(np.uint8),
    'data_info': f'Shape: {X}x{Y}x{Z}x{Ne}, Patch_radius: {PATCH_RADIUS}'
}

sio.savemat('all_denoising_results.mat', all_results)
print("✅ All results saved to 'all_denoising_results.mat'")

# ----------------------- Napari Visualization -----------------------
def create_napari_viewer():
    """Create Napari viewer with all denoising results"""
    try:
        import napari
    except ImportError:
        print("❌ Napari not installed. Install with: pip install napari[all]")
        return None
    
    print("▶ Creating Napari viewer...")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Methods Comparison")
    
    # Set consistent contrast limits
    vmin, vmax = np.percentile(mag_orig[mask_brain], [1, 99])
    
    # Add original and noisy data
    viewer.add_image(mag_orig, 
                    name='1. Original', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(mag_noisy, 
                    name='2. Noisy (σ=30)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    # Add denoised results
    viewer.add_image(deno_mag, 
                    name='3. Magnitude MP-PCA', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_split, 
                    name='4. Split MP-PCA (R/I)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_cmplx, 
                    name='5. Complex MP-PCA', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    # Add brain mask (initially hidden)
    viewer.add_image(mask_brain.astype(np.uint8), 
                    name='Brain Mask', 
                    colormap='red',
                    opacity=0.3,
                    visible=False)
    
    # Add difference maps (initially hidden)
    diff_vmax = np.percentile(np.abs(mag_noisy - mag_orig), 95)
    
    viewer.add_image(deno_mag - mag_orig,
                    name='Diff: Mag - Orig',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image(deno_split - mag_orig,
                    name='Diff: Split - Orig',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image(deno_cmplx - mag_orig,
                    name='Diff: Complex - Orig',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    print("✅ Napari viewer created successfully!")
    print("\n📖 Usage Tips:")
    print("  • Mouse wheel: Navigate through slices (Z dimension)")
    print("  • Shift + scroll: Navigate through echoes (4th dimension)")
    print("  • Toggle layer visibility with checkboxes")
    print("  • Adjust opacity with sliders")
    print("  • Enable 'Diff:' layers to see difference maps")
    
    return viewer

# ----------------------- Quick Quality Assessment -----------------------
def calculate_metrics():
    """Calculate basic quality metrics for comparison"""
    print("\n▶ Calculating quality metrics...")
    
    def snr_metric(ref, test, mask):
        """Simple SNR calculation"""
        signal = np.mean(ref[mask])
        noise = np.std((test - ref)[mask])
        return 20 * np.log10(signal / (noise + EPSILON))
    
    def ssim_metric(ref, test, mask):
        """SSIM calculation for central slice and first echo"""
        ref_slice = ref[:, :, SLICE_IDX, 0]
        test_slice = test[:, :, SLICE_IDX, 0]
        data_range = ref_slice.max() - ref_slice.min()
        if data_range > 0:
            return ssim(ref_slice, test_slice, data_range=data_range)
        return 0
    
    # Calculate metrics
    metrics = {
        'Method': ['Noisy', 'Magnitude MP-PCA', 'Split MP-PCA', 'Complex MP-PCA'],
        'SNR (dB)': [
            snr_metric(mag_orig, mag_noisy, mask_brain),
            snr_metric(mag_orig, deno_mag, mask_brain),
            snr_metric(mag_orig, deno_split, mask_brain),
            snr_metric(mag_orig, deno_cmplx, mask_brain)
        ],
        'SSIM': [
            ssim_metric(mag_orig, mag_noisy, mask_brain),
            ssim_metric(mag_orig, deno_mag, mask_brain),
            ssim_metric(mag_orig, deno_split, mask_brain),
            ssim_metric(mag_orig, deno_cmplx, mask_brain)
        ]
    }
    
    df = pd.DataFrame(metrics).round(4)
    print("\n📊 Quality Metrics Summary:")
    print("="*50)
    print(df.to_string(index=False))
    
    return df

# ----------------------- Create standalone Napari script -----------------------
def create_standalone_viewer_script():
    """Create a standalone script for viewing saved results"""
    script_content = '''
import napari
import scipy.io as sio
import numpy as np

def load_and_view_results():
    """Load saved results and create Napari viewer"""
    
    # Load data
    print("Loading saved results...")
    data = sio.loadmat('all_denoising_results.mat')
    
    mag_orig = data['mag_orig']
    mag_noisy = data['mag_noisy']
    deno_mag = data['deno_mag']
    deno_split = data['deno_split']
    deno_cmplx = data['deno_cmplx']
    mask_brain = data['mask_brain']
    
    print(f"Data shape: {mag_orig.shape}")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Results")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig[mask_brain.astype(bool)], [1, 99])
    
    # Add images
    viewer.add_image(mag_orig, name='1. Original', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy, name='2. Noisy', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag, name='3. Magnitude MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split, name='4. Split MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx, name='5. Complex MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add mask
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    print("Napari viewer ready!")
    return viewer

if __name__ == "__main__":
    viewer = load_and_view_results()
    napari.run()
'''
    
    with open('view_all_results.py', 'w') as f:
        f.write(script_content)
    
    print("✅ Standalone viewer script saved to 'view_all_results.py'")

# ----------------------- Main Execution -----------------------
if __name__ == "__main__":
    # Calculate and display metrics
    metrics_df = calculate_metrics()
    
    # Create standalone script
    create_standalone_viewer_script()
    
    # Try to create Napari viewer
    print("\n▶ Attempting to open Napari viewer...")
    try:
        viewer = create_napari_viewer()
        if viewer is not None:
            import napari
            napari.run()
        else:
            print("🔧 Napari not available. Use: python view_all_results.py")
    except Exception as e:
        print(f"❌ Error creating Napari viewer: {e}")
        print("🔧 Try running: python view_all_results.py")
    
    print("\n🎉 All denoising methods completed!")
    print("📁 Files created:")
    print("   - all_denoising_results.mat: All denoising results")
    print("   - view_all_results.py: Standalone Napari viewer")

Data dimensions: 256×224×176×6
Total number of slice-echo combinations: 1056
▶ Running Magnitude MP-PCA...


/Users/2_kumi/Desktop/complex_localpca.py:377: RuntimeWarning: invalid value encountered in divide
  denoised_arr = thetax / theta


   Magnitude MP-PCA completed in 101.3s
▶ Running Split MP-PCA (Real/Imag)...
   Split MP-PCA completed in 200.4s
▶ Running Complex MP-PCA...
   Complex MP-PCA completed in 281.6s
▶ Saving denoising results...
✅ All results saved to 'all_denoising_results.mat'

▶ Calculating quality metrics...

📊 Quality Metrics Summary:
          Method  SNR (dB)   SSIM
           Noisy   10.7390 0.8069
Magnitude MP-PCA   17.6479 0.9233
    Split MP-PCA   16.1604 0.9132
  Complex MP-PCA   17.5189 0.9291
✅ Standalone viewer script saved to 'view_all_results.py'

▶ Attempting to open Napari viewer...
▶ Creating Napari viewer...
✅ Napari viewer created successfully!

📖 Usage Tips:
  • Mouse wheel: Navigate through slices (Z dimension)
  • Shift + scroll: Navigate through echoes (4th dimension)
  • Toggle layer visibility with checkboxes
  • Adjust opacity with sliders
  • Enable 'Diff:' layers to see difference maps

🎉 All denoising methods completed!
📁 Files created:
   - all_denoising_results.mat: All 

: 

In [1]:
import numpy as np
import scipy.io as sio
import time
import pandas as pd
import matplotlib.pyplot as plt
from complex_localpca import mppca
from skimage.metrics import structural_similarity as ssim

# ----------------------- Parameters -----------------------
PATCH_RADIUS = 3        # patch radius for 7×7×7
EPSILON      = 1e-12    # to avoid divide-by-zero in SNR
SLICE_IDX    = None      # None이면 중앙 슬라이스 (Z//2) 사용

# ----------------------- Load data -----------------------
# Original complex data + brain mask
orig_mat    = sio.loadmat('meas_gre_dir1.mat')
meas_gre    = orig_mat['meas_gre']                # shape (X,Y,Z,Ne), complex
mask_brain  = orig_mat['mask_brain'].astype(bool) # shape (X,Y,Z)

# Noisy real & imag (noise case)
noise_mat   = sio.loadmat('noisy_meas_gre_dir1_30.mat')
noisy_real  = noise_mat['noisy_real'].astype(np.float32)
noisy_imag  = noise_mat['noisy_imag'].astype(np.float32)

# Dimensions
X, Y, Z, Ne = meas_gre.shape
print(f"Data dimensions: {X}×{Y}×{Z}×{Ne}")
print(f"Total number of slice-echo combinations: {Z * Ne}")

X, Y, Z, Ne = meas_gre.shape
if SLICE_IDX is None:
    SLICE_IDX = Z // 2

mag_orig  = np.abs(meas_gre)
mag_noisy = np.sqrt(noisy_real**2 + noisy_imag**2)

# ----------------------- Magnitude MP-PCA -----------------------
print("▶ Running Magnitude MP-PCA...")
t1 = time.time()
den_mag = mppca(
    mag_noisy,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Magnitude MP-PCA completed in {time.time()-t1:.1f}s")
# den_mag shape: (X, Y, Z, Ne)
deno_mag = den_mag

# ----------------------- Split MP-PCA (Real/Imag) -----------------------
print("▶ Running Split MP-PCA (Real/Imag)...")
t1 = time.time()

den_real = mppca(
    noisy_real,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)

den_imag = mppca(
    noisy_imag,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Split MP-PCA completed in {time.time()-t1:.1f}s")

# den_real, den_imag shape: (X, Y, Z, Ne)
deno_split = np.sqrt(den_real**2 + den_imag**2)

# ----------------------- Complex MP-PCA -----------------------
# build multi-channel array: real & imag concatenated
multi_ch = np.concatenate([noisy_real, noisy_imag], axis=3)  # (X,Y,Z,2*Ne)

print("▶ Running Complex MP-PCA...")
t1 = time.time()
den_all = mppca(
    multi_ch,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Complex MP-PCA completed in {time.time()-t1:.1f}s")

real_den_cmplx = den_all[..., :Ne]
imag_den_cmplx = den_all[..., Ne:]
deno_cmplx = np.abs(real_den_cmplx + 1j * imag_den_cmplx)  # (X,Y,Z,Ne)

# ----------------------- Echo averaging for visualization -----------------------
print("▶ Averaging echoes for better visualization...")

# Average across echo dimension (Ne) to get 3D volumes (X,Y,Z)
mag_orig_avg = np.mean(mag_orig, axis=3)      # (X,Y,Z)
mag_noisy_avg = np.mean(mag_noisy, axis=3)    # (X,Y,Z)
deno_mag_avg = np.mean(deno_mag, axis=3)      # (X,Y,Z)
deno_split_avg = np.mean(deno_split, axis=3)  # (X,Y,Z)
deno_cmplx_avg = np.mean(deno_cmplx, axis=3)  # (X,Y,Z)

print(f"Echo-averaged data shape: {mag_orig_avg.shape}")

# ----------------------- Save results for future use -----------------------
print("▶ Saving denoising results...")

# Save all results in one MAT file
all_results = {
    # Original 4D data (X,Y,Z,Ne)
    'mag_orig_4d': mag_orig.astype(np.float32),
    'mag_noisy_4d': mag_noisy.astype(np.float32),
    'deno_mag_4d': deno_mag.astype(np.float32),
    'deno_split_4d': deno_split.astype(np.float32),
    'deno_cmplx_4d': deno_cmplx.astype(np.float32),
    
    # Echo-averaged 3D data (X,Y,Z) for visualization
    'mag_orig_avg': mag_orig_avg.astype(np.float32),
    'mag_noisy_avg': mag_noisy_avg.astype(np.float32),
    'deno_mag_avg': deno_mag_avg.astype(np.float32),
    'deno_split_avg': deno_split_avg.astype(np.float32),
    'deno_cmplx_avg': deno_cmplx_avg.astype(np.float32),
    
    # Complex data components
    'real_den_cmplx': real_den_cmplx.astype(np.float32),
    'imag_den_cmplx': imag_den_cmplx.astype(np.float32),
    'den_real': den_real.astype(np.float32),
    'den_imag': den_imag.astype(np.float32),
    
    # Mask and metadata
    'mask_brain': mask_brain.astype(np.uint8),
    'data_info': f'Original: {X}x{Y}x{Z}x{Ne}, Averaged: {X}x{Y}x{Z}, Patch_radius: {PATCH_RADIUS}',
    'dimensions': np.array([X, Y, Z, Ne]),
    'patch_radius': PATCH_RADIUS,
    'slice_idx': SLICE_IDX
}

sio.savemat('all_denoising_results.mat', all_results)
print("✅ All results saved to 'all_denoising_results.mat'")

# Also save individual method results for easy access
individual_results = {
    'magnitude_mppca': {
        'deno_mag_4d': deno_mag.astype(np.float32),
        'deno_mag_avg': deno_mag_avg.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    },
    'split_mppca': {
        'deno_split_4d': deno_split.astype(np.float32),
        'deno_split_avg': deno_split_avg.astype(np.float32),
        'den_real': den_real.astype(np.float32),
        'den_imag': den_imag.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    },
    'complex_mppca': {
        'deno_cmplx_4d': deno_cmplx.astype(np.float32),
        'deno_cmplx_avg': deno_cmplx_avg.astype(np.float32),
        'real_den_cmplx': real_den_cmplx.astype(np.float32),
        'imag_den_cmplx': imag_den_cmplx.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    }
}

sio.savemat('magnitude_mppca_results.mat', individual_results['magnitude_mppca'])
sio.savemat('split_mppca_results.mat', individual_results['split_mppca'])
sio.savemat('complex_mppca_results.mat', individual_results['complex_mppca'])

print("✅ Individual method results also saved:")
print("   - magnitude_mppca_results.mat")
print("   - split_mppca_results.mat") 
print("   - complex_mppca_results.mat")

# ----------------------- Napari Visualization -----------------------
def create_napari_viewer():
    """Create Napari viewer with all denoising results"""
    try:
        import napari
    except ImportError:
        print("❌ Napari not installed. Install with: pip install napari[all]")
        return None
    
    print("▶ Creating Napari viewer...")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Methods Comparison")
    
    # Set consistent contrast limits using echo-averaged data
    vmin, vmax = np.percentile(mag_orig_avg[mask_brain], [1, 99])
    
    # Add echo-averaged 3D data (easier to navigate with Z-axis only)
    viewer.add_image(mag_orig_avg, 
                    name='1. Original (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(mag_noisy_avg, 
                    name='2. Noisy (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_mag_avg, 
                    name='3. Magnitude MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_split_avg, 
                    name='4. Split MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_cmplx_avg, 
                    name='5. Complex MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    # Add brain mask (initially hidden)
    viewer.add_image(mask_brain.astype(np.uint8), 
                    name='Brain Mask', 
                    colormap='red',
                    opacity=0.3,
                    visible=False)
    
    # Add difference maps for echo-averaged data (initially hidden)
    diff_vmax = np.percentile(np.abs(mag_noisy_avg - mag_orig_avg), 95)
    
    viewer.add_image(deno_mag_avg - mag_orig_avg,
                    name='Diff: Mag - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image(deno_split_avg - mag_orig_avg,
                    name='Diff: Split - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image(deno_cmplx_avg - mag_orig_avg,
                    name='Diff: Complex - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    print("✅ Napari viewer created successfully!")
    print("\n📖 Usage Tips:")
    print("  • Mouse wheel: Navigate through slices (Z dimension only)")
    print("  • Data is echo-averaged for cleaner brain visualization")
    print("  • Toggle layer visibility with checkboxes")
    print("  • Adjust opacity with sliders")
    print("  • Enable 'Diff:' layers to see difference maps")
    print("  • Original 4D data is saved in MAT files for detailed analysis")
    
    return viewer

# ----------------------- Quick Quality Assessment -----------------------
def calculate_metrics():
    """Calculate basic quality metrics for comparison"""
    print("\n▶ Calculating quality metrics...")
    
    def snr_metric(ref, test, mask):
        """Simple SNR calculation using echo-averaged data"""
        signal = np.mean(ref[mask])
        noise = np.std((test - ref)[mask])
        return 20 * np.log10(signal / (noise + EPSILON))
    
    def ssim_metric(ref, test, mask):
        """SSIM calculation for central slice using averaged data"""
        ref_slice = ref[:, :, SLICE_IDX]
        test_slice = test[:, :, SLICE_IDX]
        data_range = ref_slice.max() - ref_slice.min()
        if data_range > 0:
            return ssim(ref_slice, test_slice, data_range=data_range)
        return 0
    
    # Calculate metrics using echo-averaged data
    metrics = {
        'Method': ['Noisy', 'Magnitude MP-PCA', 'Split MP-PCA', 'Complex MP-PCA'],
        'SNR (dB)': [
            snr_metric(mag_orig_avg, mag_noisy_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_mag_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_split_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_cmplx_avg, mask_brain)
        ],
        'SSIM': [
            ssim_metric(mag_orig_avg, mag_noisy_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_mag_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_split_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_cmplx_avg, mask_brain)
        ]
    }
    
    df = pd.DataFrame(metrics).round(4)
    print("\n📊 Quality Metrics Summary (Echo-averaged):")
    print("="*55)
    print(df.to_string(index=False))
    
    return df

# ----------------------- Create standalone Napari script -----------------------
def create_standalone_viewer_script():
    """Create a standalone script for viewing saved results"""
    script_content = '''
import napari
import scipy.io as sio
import numpy as np

def load_and_view_results():
    """Load saved results and create Napari viewer"""
    
    # Load data
    print("Loading saved results...")
    data = sio.loadmat('all_denoising_results.mat')
    
    # Echo-averaged 3D data for clean visualization
    mag_orig_avg = data['mag_orig_avg']
    mag_noisy_avg = data['mag_noisy_avg']
    deno_mag_avg = data['deno_mag_avg']
    deno_split_avg = data['deno_split_avg']
    deno_cmplx_avg = data['deno_cmplx_avg']
    mask_brain = data['mask_brain']
    
    print(f"Echo-averaged data shape: {mag_orig_avg.shape}")
    print(f"Original 4D data also available in MAT file")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Results (Echo-averaged)")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig_avg[mask_brain.astype(bool)], [1, 99])
    
    # Add echo-averaged images
    viewer.add_image(mag_orig_avg, name='1. Original', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_avg, name='2. Noisy', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_avg, name='3. Magnitude MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_avg, name='4. Split MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_avg, name='5. Complex MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add mask
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    # Add difference maps
    diff_vmax = np.percentile(np.abs(mag_noisy_avg - mag_orig_avg), 95)
    viewer.add_image(deno_mag_avg - mag_orig_avg, name='Diff: Mag - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image(deno_split_avg - mag_orig_avg, name='Diff: Split - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image(deno_cmplx_avg - mag_orig_avg, name='Diff: Complex - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    
    print("Napari viewer ready!")
    print("Navigate through Z-slices with mouse wheel")
    return viewer

def load_4d_data():
    """Load original 4D data for detailed analysis"""
    print("Loading 4D data...")
    data = sio.loadmat('all_denoising_results.mat')
    
    # Create viewer for 4D data
    viewer = napari.Viewer(title="MP-PCA Denoising Results (4D - All Echoes)")
    
    # Original 4D data
    mag_orig_4d = data['mag_orig_4d']
    mag_noisy_4d = data['mag_noisy_4d']
    deno_mag_4d = data['deno_mag_4d']
    deno_split_4d = data['deno_split_4d']
    deno_cmplx_4d = data['deno_cmplx_4d']
    mask_brain = data['mask_brain']
    
    vmin, vmax = np.percentile(mag_orig_4d[mask_brain.astype(bool)], [1, 99])
    
    viewer.add_image(mag_orig_4d, name='1. Original (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_4d, name='2. Noisy (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_4d, name='3. Magnitude MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_4d, name='4. Split MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_4d, name='5. Complex MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    
    print("4D Napari viewer ready!")
    print("Use Shift+scroll to navigate through echoes")
    return viewer

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "4d":
        viewer = load_4d_data()
    else:
        viewer = load_and_view_results()
    
    napari.run()
'''
    
    with open('view_all_results.py', 'w') as f:
        f.write(script_content)
    
    print("✅ Standalone viewer script saved to 'view_all_results.py'")
    print("   Usage: python view_all_results.py      # Echo-averaged 3D view")
    print("          python view_all_results.py 4d   # Original 4D view")

# ----------------------- Main Execution -----------------------
if __name__ == "__main__":
    # Calculate and display metrics
    metrics_df = calculate_metrics()
    
    # Create standalone script
    create_standalone_viewer_script()
    
    # Try to create Napari viewer
    print("\n▶ Attempting to open Napari viewer...")
    try:
        viewer = create_napari_viewer()
        if viewer is not None:
            import napari
            napari.run()
        else:
            print("🔧 Napari not available. Use: python view_all_results.py")
    except Exception as e:
        print(f"❌ Error creating Napari viewer: {e}")
        print("🔧 Try running: python view_all_results.py")
    
    print("\n🎉 All denoising methods completed!")
    print("\n📁 Files created:")
    print("   - all_denoising_results.mat: Complete results (4D + echo-averaged)")
    print("   - magnitude_mppca_results.mat: Magnitude domain results")
    print("   - split_mppca_results.mat: Split domain results")
    print("   - complex_mppca_results.mat: Complex domain results")
    print("   - view_all_results.py: Standalone Napari viewer")
    print("\n🎯 Visualization options:")
    print("   - python view_all_results.py      # Clean 3D view (echo-averaged)")
    print("   - python view_all_results.py 4d   # Full 4D view (all echoes)")
    print("\n💡 MAT file contents:")
    print("   - *_4d: Original 4D data (X,Y,Z,Ne)")
    print("   - *_avg: Echo-averaged 3D data (X,Y,Z)")
    print("   - mask_brain: Brain mask")
    print("   - Individual method files for external use")

Data dimensions: 256×224×176×6
Total number of slice-echo combinations: 1056
▶ Running Magnitude MP-PCA...


/Users/2_kumi/Desktop/complex_localpca.py:377: RuntimeWarning: invalid value encountered in divide
  denoised_arr = thetax / theta


   Magnitude MP-PCA completed in 99.9s
▶ Running Split MP-PCA (Real/Imag)...
   Split MP-PCA completed in 207.5s
▶ Running Complex MP-PCA...
   Complex MP-PCA completed in 324.6s
▶ Averaging echoes for better visualization...
Echo-averaged data shape: (256, 224, 176)
▶ Saving denoising results...
✅ All results saved to 'all_denoising_results.mat'
✅ Individual method results also saved:
   - magnitude_mppca_results.mat
   - split_mppca_results.mat
   - complex_mppca_results.mat

▶ Calculating quality metrics...

📊 Quality Metrics Summary (Echo-averaged):
          Method  SNR (dB)   SSIM
           Noisy   17.7103 0.8911
Magnitude MP-PCA   19.4660 0.9416
    Split MP-PCA   19.7435 0.9259
  Complex MP-PCA   20.0765 0.9314
✅ Standalone viewer script saved to 'view_all_results.py'
   Usage: python view_all_results.py      # Echo-averaged 3D view
          python view_all_results.py 4d   # Original 4D view

▶ Attempting to open Napari viewer...
▶ Creating Napari viewer...
✅ Napari viewer cr

In [4]:
import numpy as np
import scipy.io as sio
import time
import pandas as pd
import matplotlib.pyplot as plt
from complex_localpca import mppca
from skimage.metrics import structural_similarity as ssim

# ----------------------- Parameters -----------------------
PATCH_RADIUS = 3        # patch radius for 7×7×7
EPSILON      = 1e-12    # to avoid divide-by-zero in SNR
SLICE_IDX    = None      # None이면 중앙 슬라이스 (Z//2) 사용

# ----------------------- Load data -----------------------
# Original complex data + brain mask
orig_mat    = sio.loadmat('meas_gre_dir1.mat')
meas_gre    = orig_mat['meas_gre']                # shape (X,Y,Z,Ne), complex
mask_brain  = orig_mat['mask_brain'].astype(bool) # shape (X,Y,Z)

# Noisy real & imag (noise case)
noise_mat   = sio.loadmat('noisy_meas_gre_dir1_30.mat')
noisy_real  = noise_mat['noisy_real'].astype(np.float32)
noisy_imag  = noise_mat['noisy_imag'].astype(np.float32)

# Dimensions
X, Y, Z, Ne = meas_gre.shape
print(f"Data dimensions: {X}×{Y}×{Z}×{Ne}")
print(f"Total number of slice-echo combinations: {Z * Ne}")

X, Y, Z, Ne = meas_gre.shape
if SLICE_IDX is None:
    SLICE_IDX = Z // 2

mag_orig  = np.abs(meas_gre)
mag_noisy = np.sqrt(noisy_real**2 + noisy_imag**2)

# ----------------------- Magnitude MP-PCA -----------------------
print("▶ Running Magnitude MP-PCA...")
t1 = time.time()
den_mag = mppca(
    mag_noisy,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Magnitude MP-PCA completed in {time.time()-t1:.1f}s")
# den_mag shape: (X, Y, Z, Ne)
deno_mag = den_mag

# ----------------------- Split MP-PCA (Real/Imag) -----------------------
print("▶ Running Split MP-PCA (Real/Imag)...")
t1 = time.time()

den_real = mppca(
    noisy_real,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)

den_imag = mppca(
    noisy_imag,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Split MP-PCA completed in {time.time()-t1:.1f}s")

# den_real, den_imag shape: (X, Y, Z, Ne)
deno_split = np.sqrt(den_real**2 + den_imag**2)

# ----------------------- Complex MP-PCA -----------------------
# build multi-channel array: real & imag concatenated
multi_ch = np.concatenate([noisy_real, noisy_imag], axis=3)  # (X,Y,Z,2*Ne)

print("▶ Running Complex MP-PCA...")
t1 = time.time()
den_all = mppca(
    multi_ch,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method='svd',
    return_sigma=False
)
print(f"   Complex MP-PCA completed in {time.time()-t1:.1f}s")

real_den_cmplx = den_all[..., :Ne]
imag_den_cmplx = den_all[..., Ne:]
deno_cmplx = np.abs(real_den_cmplx + 1j * imag_den_cmplx)  # (X,Y,Z,Ne)

# ----------------------- Echo averaging for visualization -----------------------
print("▶ Averaging echoes for better visualization...")

# Average across echo dimension (Ne) to get 3D volumes (X,Y,Z)
mag_orig_avg = np.mean(mag_orig, axis=3)      # (X,Y,Z)
mag_noisy_avg = np.mean(mag_noisy, axis=3)    # (X,Y,Z)
deno_mag_avg = np.mean(deno_mag, axis=3)      # (X,Y,Z)
deno_split_avg = np.mean(deno_split, axis=3)  # (X,Y,Z)
deno_cmplx_avg = np.mean(deno_cmplx, axis=3)  # (X,Y,Z)

print(f"Echo-averaged data shape before transpose: {mag_orig_avg.shape}")

# Transpose to get proper axial view (swap axes for correct orientation)
# Original: (X,Y,Z) -> Transpose to (Z,Y,X) for axial slices
mag_orig_avg = np.transpose(mag_orig_avg, (2, 1, 0))      # (Z,Y,X)
mag_noisy_avg = np.transpose(mag_noisy_avg, (2, 1, 0))    # (Z,Y,X)
deno_mag_avg = np.transpose(deno_mag_avg, (2, 1, 0))      # (Z,Y,X)
deno_split_avg = np.transpose(deno_split_avg, (2, 1, 0))  # (Z,Y,X)
deno_cmplx_avg = np.transpose(deno_cmplx_avg, (2, 1, 0))  # (Z,Y,X)
mask_brain_transposed = np.transpose(mask_brain, (2, 1, 0))  # (Z,Y,X)

print(f"Echo-averaged data shape after transpose: {mag_orig_avg.shape}")
print(f"Now Z-axis (slice direction) is the first dimension")

# ----------------------- Save results for future use -----------------------
print("▶ Saving denoising results...")

# Save all results in one MAT file
all_results = {
    # Original 4D data (X,Y,Z,Ne)
    'mag_orig_4d': mag_orig.astype(np.float32),
    'mag_noisy_4d': mag_noisy.astype(np.float32),
    'deno_mag_4d': deno_mag.astype(np.float32),
    'deno_split_4d': deno_split.astype(np.float32),
    'deno_cmplx_4d': deno_cmplx.astype(np.float32),
    
    # Echo-averaged 3D data (X,Y,Z) for visualization
    'mag_orig_avg': mag_orig_avg.astype(np.float32),
    'mag_noisy_avg': mag_noisy_avg.astype(np.float32),
    'deno_mag_avg': deno_mag_avg.astype(np.float32),
    'deno_split_avg': deno_split_avg.astype(np.float32),
    'deno_cmplx_avg': deno_cmplx_avg.astype(np.float32),
    'mask_brain_transposed': mask_brain_transposed.astype(np.uint8),
    
    # Complex data components
    'real_den_cmplx': real_den_cmplx.astype(np.float32),
    'imag_den_cmplx': imag_den_cmplx.astype(np.float32),
    'den_real': den_real.astype(np.float32),
    'den_imag': den_imag.astype(np.float32),
    
    # Mask and metadata
    'mask_brain': mask_brain.astype(np.uint8),
    'data_info': f'Original: {X}x{Y}x{Z}x{Ne}, Averaged: {X}x{Y}x{Z}, Patch_radius: {PATCH_RADIUS}',
    'dimensions': np.array([X, Y, Z, Ne]),
    'patch_radius': PATCH_RADIUS,
    'slice_idx': SLICE_IDX
}

sio.savemat('all_denoising_results.mat', all_results)
print("✅ All results saved to 'all_denoising_results.mat'")

# Also save individual method results for easy access
individual_results = {
    'magnitude_mppca': {
        'deno_mag_4d': deno_mag.astype(np.float32),
        'deno_mag_avg': deno_mag_avg.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    },
    'split_mppca': {
        'deno_split_4d': deno_split.astype(np.float32),
        'deno_split_avg': deno_split_avg.astype(np.float32),
        'den_real': den_real.astype(np.float32),
        'den_imag': den_imag.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    },
    'complex_mppca': {
        'deno_cmplx_4d': deno_cmplx.astype(np.float32),
        'deno_cmplx_avg': deno_cmplx_avg.astype(np.float32),
        'real_den_cmplx': real_den_cmplx.astype(np.float32),
        'imag_den_cmplx': imag_den_cmplx.astype(np.float32),
        'mask_brain': mask_brain.astype(np.uint8)
    }
}

sio.savemat('magnitude_mppca_results.mat', individual_results['magnitude_mppca'])
sio.savemat('split_mppca_results.mat', individual_results['split_mppca'])
sio.savemat('complex_mppca_results.mat', individual_results['complex_mppca'])

print("✅ Individual method results also saved:")
print("   - magnitude_mppca_results.mat")
print("   - split_mppca_results.mat") 
print("   - complex_mppca_results.mat")

# ----------------------- Napari Visualization -----------------------
def create_napari_viewer():
    """Create Napari viewer with all denoising results"""
    try:
        import napari
    except ImportError:
        print("❌ Napari not installed. Install with: pip install napari[all]")
        return None
    
    print("▶ Creating Napari viewer...")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Methods Comparison")
    
    # Debug: Print data shapes and types
    print(f"Data shapes - Original: {mag_orig_avg.shape}, Mask: {mask_brain_transposed.shape}")
    print(f"Data types - Original: {mag_orig_avg.dtype}, Mask: {mask_brain_transposed.dtype}")
    
    # Set consistent contrast limits using echo-averaged data
    vmin, vmax = np.percentile(mag_orig_avg[mask_brain_transposed.astype(bool)], [1, 99])
    
    # Add echo-averaged 3D data (easier to navigate with Z-axis only)
    # Ensure all data is float32 and has correct shape
    viewer.add_image(mag_orig_avg.astype(np.float32), 
                    name='1. Original (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(mag_noisy_avg.astype(np.float32), 
                    name='2. Noisy (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_mag_avg.astype(np.float32), 
                    name='3. Magnitude MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_split_avg.astype(np.float32), 
                    name='4. Split MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    viewer.add_image(deno_cmplx_avg.astype(np.float32), 
                    name='5. Complex MP-PCA (avg)', 
                    colormap='gray',
                    contrast_limits=[vmin, vmax])
    
    # Add brain mask separately with correct dtype (initially hidden)
    viewer.add_image(mask_brain_transposed.astype(np.float32),  # Use transposed mask
                    name='Brain Mask', 
                    colormap='red',
                    opacity=0.3,
                    visible=False)
    
    # Add difference maps for echo-averaged data (initially hidden)
    diff_vmax = np.percentile(np.abs(mag_noisy_avg - mag_orig_avg), 95)
    
    viewer.add_image((deno_mag_avg - mag_orig_avg).astype(np.float32),
                    name='Diff: Mag - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image((deno_split_avg - mag_orig_avg).astype(np.float32),
                    name='Diff: Split - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    viewer.add_image((deno_cmplx_avg - mag_orig_avg).astype(np.float32),
                    name='Diff: Complex - Orig (avg)',
                    colormap='bwr',
                    contrast_limits=[-diff_vmax, diff_vmax],
                    visible=False)
    
    print("✅ Napari viewer created successfully!")
    print("\n📖 Usage Tips:")
    print("  • Mouse wheel: Navigate through slices (Z dimension only)")
    print("  • Data is echo-averaged for cleaner brain visualization")
    print("  • Toggle layer visibility with checkboxes")
    print("  • Adjust opacity with sliders")
    print("  • Enable 'Diff:' layers to see difference maps")
    print("  • Original 4D data is saved in MAT files for detailed analysis")
    
    return viewer

# ----------------------- Quick Quality Assessment -----------------------
def calculate_metrics():
    """Calculate basic quality metrics for comparison"""
    print("\n▶ Calculating quality metrics...")
    
    def snr_metric(ref, test, mask):
        """Simple SNR calculation using echo-averaged data"""
        signal = np.mean(ref[mask])
        noise = np.std((test - ref)[mask])
        return 20 * np.log10(signal / (noise + EPSILON))
    
    def ssim_metric(ref, test, mask):
        """SSIM calculation for central slice using averaged data"""
        ref_slice = ref[:, :, SLICE_IDX]
        test_slice = test[:, :, SLICE_IDX]
        data_range = ref_slice.max() - ref_slice.min()
        if data_range > 0:
            return ssim(ref_slice, test_slice, data_range=data_range)
        return 0
    
    # Calculate metrics using echo-averaged data
    metrics = {
        'Method': ['Noisy', 'Magnitude MP-PCA', 'Split MP-PCA', 'Complex MP-PCA'],
        'SNR (dB)': [
            snr_metric(mag_orig_avg, mag_noisy_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_mag_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_split_avg, mask_brain),
            snr_metric(mag_orig_avg, deno_cmplx_avg, mask_brain)
        ],
        'SSIM': [
            ssim_metric(mag_orig_avg, mag_noisy_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_mag_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_split_avg, mask_brain),
            ssim_metric(mag_orig_avg, deno_cmplx_avg, mask_brain)
        ]
    }
    
    df = pd.DataFrame(metrics).round(4)
    print("\n📊 Quality Metrics Summary (Echo-averaged):")
    print("="*55)
    print(df.to_string(index=False))
    
    return df

# ----------------------- Create standalone Napari script -----------------------
def create_standalone_viewer_script():
    """Create a standalone script for viewing saved results"""
    script_content = '''
import napari
import scipy.io as sio
import numpy as np

def load_and_view_results():
    """Load saved results and create Napari viewer"""
    
    # Load data
    print("Loading saved results...")
    data = sio.loadmat('all_denoising_results.mat')
    
    # Echo-averaged 3D data for clean visualization
    mag_orig_avg = data['mag_orig_avg'].astype(np.float32)
    mag_noisy_avg = data['mag_noisy_avg'].astype(np.float32)
    deno_mag_avg = data['deno_mag_avg'].astype(np.float32)
    deno_split_avg = data['deno_split_avg'].astype(np.float32)
    deno_cmplx_avg = data['deno_cmplx_avg'].astype(np.float32)
    
    # Use transposed mask if available, otherwise transpose original mask
    if 'mask_brain_transposed' in data:
        mask_brain = data['mask_brain_transposed'].astype(np.float32)
    else:
        mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
    
    print(f"Echo-averaged data shape: {mag_orig_avg.shape}")
    print(f"Data type: {mag_orig_avg.dtype}")
    print(f"Z-axis range: 0 to {mag_orig_avg.shape[0]-1} (axial slices)")
    print(f"Original 4D data also available in MAT file")
    
    # Create viewer
    viewer = napari.Viewer(title="MP-PCA Denoising Results (Axial View)")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig_avg[mask_brain.astype(bool)], [1, 99])
    
    # Add echo-averaged images - ensure all are float32
    viewer.add_image(mag_orig_avg, name='1. Original', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_avg, name='2. Noisy', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_avg, name='3. Magnitude MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_avg, name='4. Split MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_avg, name='5. Complex MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add mask
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    # Add difference maps
    diff_vmax = np.percentile(np.abs(mag_noisy_avg - mag_orig_avg), 95)
    viewer.add_image((deno_mag_avg - mag_orig_avg).astype(np.float32), name='Diff: Mag - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_split_avg - mag_orig_avg).astype(np.float32), name='Diff: Split - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_cmplx_avg - mag_orig_avg).astype(np.float32), name='Diff: Complex - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    
    print("Napari viewer ready!")
    print(f"Navigate through axial slices (0 to {mag_orig_avg.shape[0]-1}) with mouse wheel")
    return viewer

def load_4d_data():
    """Load original 4D data for detailed analysis"""
    print("Loading 4D data...")
    data = sio.loadmat('all_denoising_results.mat')
    
    # Create viewer for 4D data
    viewer = napari.Viewer(title="MP-PCA Denoising Results (4D - All Echoes)")
    
    # Original 4D data - ensure float32 and transpose for axial view
    mag_orig_4d = data['mag_orig_4d'].astype(np.float32)
    mag_noisy_4d = data['mag_noisy_4d'].astype(np.float32)
    deno_mag_4d = data['deno_mag_4d'].astype(np.float32)
    deno_split_4d = data['deno_split_4d'].astype(np.float32)
    deno_cmplx_4d = data['deno_cmplx_4d'].astype(np.float32)
    
    # Transpose 4D data from (X,Y,Z,Ne) to (Z,Y,X,Ne) for axial view
    mag_orig_4d = np.transpose(mag_orig_4d, (2, 1, 0, 3))
    mag_noisy_4d = np.transpose(mag_noisy_4d, (2, 1, 0, 3))
    deno_mag_4d = np.transpose(deno_mag_4d, (2, 1, 0, 3))
    deno_split_4d = np.transpose(deno_split_4d, (2, 1, 0, 3))
    deno_cmplx_4d = np.transpose(deno_cmplx_4d, (2, 1, 0, 3))
    
    mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
    
    print(f"4D data shape: {mag_orig_4d.shape} (Z,Y,X,Ne)")
    print(f"Z-axis range: 0 to {mag_orig_4d.shape[0]-1} (axial slices)")
    print(f"Echo range: 0 to {mag_orig_4d.shape[3]-1}")
    
    vmin, vmax = np.percentile(mag_orig_4d[mask_brain.astype(bool)], [1, 99])
    
    viewer.add_image(mag_orig_4d, name='1. Original (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_4d, name='2. Noisy (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_4d, name='3. Magnitude MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_4d, name='4. Split MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_4d, name='5. Complex MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    
    print("4D Napari viewer ready!")
    print("Use mouse wheel for axial slices, Shift+scroll for echoes")
    return viewer

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "4d":
        viewer = load_4d_data()
    else:
        viewer = load_and_view_results()
    
    napari.run()
'''
    
    with open('view_all_results.py', 'w') as f:
        f.write(script_content)
    
    print("✅ Standalone viewer script saved to 'view_all_results.py'")
    print("   Usage: python view_all_results.py      # Echo-averaged 3D view")
    print("          python view_all_results.py 4d   # Original 4D view")

# ----------------------- Main Execution -----------------------
if __name__ == "__main__":
    # Calculate and display metrics
    metrics_df = calculate_metrics()
    
    # Create standalone script
    create_standalone_viewer_script()
    
    # Try to create Napari viewer
    print("\n▶ Attempting to open Napari viewer...")
    try:
        viewer = create_napari_viewer()
        if viewer is not None:
            import napari
            napari.run()
        else:
            print("🔧 Napari not available. Use: python view_all_results.py")
    except Exception as e:
        print(f"❌ Error creating Napari viewer: {e}")
        print("🔧 Try running: python view_all_results.py")
    
    print("\n🎉 All denoising methods completed!")
    print("\n📁 Files created:")
    print("   - all_denoising_results.mat: Complete results (4D + echo-averaged)")
    print("   - magnitude_mppca_results.mat: Magnitude domain results")
    print("   - split_mppca_results.mat: Split domain results")
    print("   - complex_mppca_results.mat: Complex domain results")
    print("   - view_all_results.py: Standalone Napari viewer")
    print("\n🎯 Visualization options:")
    print("   - python view_all_results.py      # Clean 3D view (echo-averaged)")
    print("   - python view_all_results.py 4d   # Full 4D view (all echoes)")
    print("\n💡 MAT file contents:")
    print("   - *_4d: Original 4D data (X,Y,Z,Ne)")
    print("   - *_avg: Echo-averaged 3D data (X,Y,Z)")
    print("   - mask_brain: Brain mask")
    print("   - Individual method files for external use")

Data dimensions: 256×224×176×6
Total number of slice-echo combinations: 1056
▶ Running Magnitude MP-PCA...


/Users/2_kumi/Desktop/complex_localpca.py:377: RuntimeWarning: invalid value encountered in divide
  denoised_arr = thetax / theta


   Magnitude MP-PCA completed in 95.2s
▶ Running Split MP-PCA (Real/Imag)...
   Split MP-PCA completed in 196.2s
▶ Running Complex MP-PCA...
   Complex MP-PCA completed in 188.5s
▶ Averaging echoes for better visualization...
Echo-averaged data shape before transpose: (256, 224, 176)
Echo-averaged data shape after transpose: (176, 224, 256)
Now Z-axis (slice direction) is the first dimension
▶ Saving denoising results...
✅ All results saved to 'all_denoising_results.mat'
✅ Individual method results also saved:
   - magnitude_mppca_results.mat
   - split_mppca_results.mat
   - complex_mppca_results.mat

▶ Calculating quality metrics...


IndexError: boolean index did not match indexed array along dimension 0; dimension is 176 but corresponding boolean dimension is 256

In [7]:
import napari
import scipy.io as sio
import numpy as np

def load_and_view():
    """Load all_denoising_results.mat and view in Napari with correct axis orientation"""
    
    print("📁 Loading all_denoising_results.mat...")
    
    # Load the data
    data = sio.loadmat('all_denoising_results.mat')
    
    # Check what's available
    print("📊 Available data in MAT file:")
    for key in data.keys():
        if not key.startswith('__'):
            if isinstance(data[key], np.ndarray):
                print(f"  {key}: {data[key].shape}")
            else:
                print(f"  {key}: {type(data[key])}")
    
    # Load the data we need
    if 'mag_orig_avg' in data:
        # We already have echo-averaged data
        print("\n✅ Using pre-computed echo-averaged data")
        mag_orig = data['mag_orig_avg'].astype(np.float32)
        mag_noisy = data['mag_noisy_avg'].astype(np.float32)
        deno_mag = data['deno_mag_avg'].astype(np.float32)
        deno_split = data['deno_split_avg'].astype(np.float32)
        deno_cmplx = data['deno_cmplx_avg'].astype(np.float32)
        
        print(f"📐 Data shape: {mag_orig.shape}")
        
        # Check if data is already transposed for axial view
        if mag_orig.shape[0] == 176:  # Z dimension is first -> already transposed
            print("✅ Data already in axial orientation (Z,Y,X)")
            # Use transposed mask if available
            if 'mask_brain_transposed' in data:
                mask_brain = data['mask_brain_transposed'].astype(np.float32)
            else:
                mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
        else:  # Data needs to be transposed
            print("🔄 Transposing data for axial view...")
            mag_orig = np.transpose(mag_orig, (2, 1, 0))
            mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
            deno_mag = np.transpose(deno_mag, (2, 1, 0))
            deno_split = np.transpose(deno_split, (2, 1, 0))
            deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
            mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
            
    else:
        # Use 4D data and average echoes
        print("\n🔄 Processing 4D data...")
        mag_orig_4d = data['mag_orig_4d']
        mag_noisy_4d = data['mag_noisy_4d'] 
        deno_mag_4d = data['deno_mag_4d']
        deno_split_4d = data['deno_split_4d']
        deno_cmplx_4d = data['deno_cmplx_4d']
        mask_brain = data['mask_brain']
        
        # Average across echo dimension
        mag_orig = np.mean(mag_orig_4d, axis=3).astype(np.float32)
        mag_noisy = np.mean(mag_noisy_4d, axis=3).astype(np.float32)
        deno_mag = np.mean(deno_mag_4d, axis=3).astype(np.float32)
        deno_split = np.mean(deno_split_4d, axis=3).astype(np.float32)
        deno_cmplx = np.mean(deno_cmplx_4d, axis=3).astype(np.float32)
        
        # Transpose all data for axial view: (X,Y,Z) -> (Z,Y,X)
        print("🔄 Transposing axes for axial view...")
        mag_orig = np.transpose(mag_orig, (2, 1, 0))
        mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
        deno_mag = np.transpose(deno_mag, (2, 1, 0))
        deno_split = np.transpose(deno_split, (2, 1, 0))
        deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
        mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
    
    print(f"\n📐 Final data shape: {mag_orig.shape}")
    print(f"📐 Mask shape: {mask_brain.shape}")
    print(f"📐 Axial slices: 0 to {mag_orig.shape[0]-1}")
    
    # Verify dimensions match
    if mag_orig.shape != mask_brain.shape:
        print("⚠️  Warning: Data and mask shapes don't match!")
        print(f"   Data shape: {mag_orig.shape}")
        print(f"   Mask shape: {mask_brain.shape}")
        return None
    
    # Create Napari viewer
    print("\n🎯 Creating Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - Axial View")
    
    # Set contrast limits
    print("📊 Computing contrast limits...")
    try:
        vmin, vmax = np.percentile(mag_orig[mask_brain.astype(bool)], [1, 99])
        print(f"📊 Contrast limits: {vmin:.1f} - {vmax:.1f}")
    except Exception as e:
        print(f"⚠️  Error computing contrast limits: {e}")
        print("   Using data min/max instead")
        vmin, vmax = np.percentile(mag_orig, [1, 99])
        print(f"📊 Fallback contrast limits: {vmin:.1f} - {vmax:.1f}")
    
    # Add main images
    viewer.add_image(mag_orig, name='1. Original', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy, name='2. Noisy', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag, name='3. Magnitude MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split, name='4. Split MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx, name='5. Complex MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add brain mask (hidden by default)
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    # Add difference maps (hidden by default)
    diff_vmax = np.percentile(np.abs(mag_noisy - mag_orig), 95)
    
    viewer.add_image((deno_mag - mag_orig), name='Diff: Mag - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_split - mag_orig), name='Diff: Split - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_cmplx - mag_orig), name='Diff: Complex - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    
    print("✅ Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Checkboxes: Show/hide layers") 
    print("  • Sliders: Adjust opacity")
    print("  • Enable 'Diff:' layers to see difference maps")
    
    return viewer

def load_4d_view():
    """Load 4D data for detailed echo-by-echo analysis"""
    
    print("📁 Loading 4D data from all_denoising_results.mat...")
    
    data = sio.loadmat('all_denoising_results.mat')
    
    # Load 4D data
    mag_orig_4d = data['mag_orig_4d'].astype(np.float32)
    mag_noisy_4d = data['mag_noisy_4d'].astype(np.float32)
    deno_mag_4d = data['deno_mag_4d'].astype(np.float32)
    deno_split_4d = data['deno_split_4d'].astype(np.float32)
    deno_cmplx_4d = data['deno_cmplx_4d'].astype(np.float32)
    mask_brain = data['mask_brain']
    
    # Transpose for axial view: (X,Y,Z,Ne) -> (Z,Y,X,Ne)
    print("🔄 Transposing 4D data for axial view...")
    mag_orig_4d = np.transpose(mag_orig_4d, (2, 1, 0, 3))
    mag_noisy_4d = np.transpose(mag_noisy_4d, (2, 1, 0, 3))
    deno_mag_4d = np.transpose(deno_mag_4d, (2, 1, 0, 3))
    deno_split_4d = np.transpose(deno_split_4d, (2, 1, 0, 3))
    deno_cmplx_4d = np.transpose(deno_cmplx_4d, (2, 1, 0, 3))
    mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
    
    print(f"📐 4D data shape: {mag_orig_4d.shape} (Z,Y,X,Echo)")
    print(f"📐 Axial slices: 0 to {mag_orig_4d.shape[0]-1}")
    print(f"📐 Echoes: 0 to {mag_orig_4d.shape[3]-1}")
    
    # Create Napari viewer
    print("\n🎯 Creating 4D Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - 4D View (All Echoes)")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig_4d[mask_brain.astype(bool)], [1, 99])
    
    # Add 4D images
    viewer.add_image(mag_orig_4d, name='1. Original (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_4d, name='2. Noisy (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_4d, name='3. Magnitude MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_4d, name='4. Split MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_4d, name='5. Complex MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    
    print("✅ 4D Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Shift + scroll: Navigate through echoes")
    
    return viewer

if __name__ == "__main__":
    import sys
    
    print("🚀 MAT Data Napari Viewer")
    print("=" * 30)
    
    if len(sys.argv) > 1 and sys.argv[1] == "4d":
        print("📊 Loading 4D view (all echoes)...")
        viewer = load_4d_view()
    else:
        print("📊 Loading 3D view (echo-averaged)...")
        viewer = load_and_view()
    
    if viewer is not None:
        napari.run()
    else:
        print("❌ Failed to create viewer")

🚀 MAT Data Napari Viewer
📊 Loading 3D view (echo-averaged)...
📁 Loading all_denoising_results.mat...
📊 Available data in MAT file:
  mag_orig_4d: (256, 224, 176, 6)
  mag_noisy_4d: (256, 224, 176, 6)
  deno_mag_4d: (256, 224, 176, 6)
  deno_split_4d: (256, 224, 176, 6)
  deno_cmplx_4d: (256, 224, 176, 6)
  mag_orig_avg: (176, 224, 256)
  mag_noisy_avg: (176, 224, 256)
  deno_mag_avg: (176, 224, 256)
  deno_split_avg: (176, 224, 256)
  deno_cmplx_avg: (176, 224, 256)
  mask_brain_transposed: (176, 224, 256)
  real_den_cmplx: (256, 224, 176, 6)
  imag_den_cmplx: (256, 224, 176, 6)
  den_real: (256, 224, 176, 6)
  den_imag: (256, 224, 176, 6)
  mask_brain: (256, 224, 176)
  data_info: (1,)
  dimensions: (1, 4)
  patch_radius: (1, 1)
  slice_idx: (1, 1)

✅ Using pre-computed echo-averaged data
📐 Data shape: (176, 224, 256)
✅ Data already in axial orientation (Z,Y,X)

📐 Final data shape: (176, 224, 256)
📐 Mask shape: (176, 224, 256)
📐 Axial slices: 0 to 175

🎯 Creating Napari viewer...
📊 Co

In [6]:
import napari
import scipy.io as sio
import numpy as np

def load_and_view():
    """Load all_denoising_results.mat and view in Napari with correct axis orientation"""
    
    print("📁 Loading all_denoising_results.mat...")
    
    # Load the data
    data = sio.loadmat('all_denoising_results.mat')
    
    # Check what's available
    print("📊 Available data in MAT file:")
    for key in data.keys():
        if not key.startswith('__'):
            if isinstance(data[key], np.ndarray):
                print(f"  {key}: {data[key].shape}")
            else:
                print(f"  {key}: {type(data[key])}")
    
    # Load the data we need
    if 'mag_orig_avg' in data:
        # We already have echo-averaged data
        print("\n✅ Using pre-computed echo-averaged data")
        mag_orig = data['mag_orig_avg'].astype(np.float32)
        mag_noisy = data['mag_noisy_avg'].astype(np.float32)
        deno_mag = data['deno_mag_avg'].astype(np.float32)
        deno_split = data['deno_split_avg'].astype(np.float32)
        deno_cmplx = data['deno_cmplx_avg'].astype(np.float32)
        
        print(f"📐 Data shape: {mag_orig.shape}")
        
        # Check if data is already transposed for axial view
        if mag_orig.shape[0] == 176:  # Z dimension is first -> already transposed
            print("✅ Data already in axial orientation (Z,Y,X)")
            # Use transposed mask if available
            if 'mask_brain_transposed' in data:
                mask_brain = data['mask_brain_transposed'].astype(np.float32)
            else:
                mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
        else:  # Data needs to be transposed
            print("🔄 Transposing data for axial view...")
            mag_orig = np.transpose(mag_orig, (2, 1, 0))
            mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
            deno_mag = np.transpose(deno_mag, (2, 1, 0))
            deno_split = np.transpose(deno_split, (2, 1, 0))
            deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
            mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
            
        # Rotate images 90 degrees clockwise for better viewing
        print("🔄 Rotating images 90° clockwise...")
        mag_orig = np.rot90(mag_orig, k=-1, axes=(1, 2))      # k=-1 for clockwise
        mag_noisy = np.rot90(mag_noisy, k=-1, axes=(1, 2))
        deno_mag = np.rot90(deno_mag, k=-1, axes=(1, 2))
        deno_split = np.rot90(deno_split, k=-1, axes=(1, 2))
        deno_cmplx = np.rot90(deno_cmplx, k=-1, axes=(1, 2))
        mask_brain = np.rot90(mask_brain, k=-1, axes=(1, 2))
            mag_orig = np.transpose(mag_orig, (2, 1, 0))
            mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
            deno_mag = np.transpose(deno_mag, (2, 1, 0))
            deno_split = np.transpose(deno_split, (2, 1, 0))
            deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
            mask_brain = np.transpose(data['mask_brain'], (2, 1, 0)).astype(np.float32)
            
    else:
        # Use 4D data and average echoes
        print("\n🔄 Processing 4D data...")
        mag_orig_4d = data['mag_orig_4d']
        mag_noisy_4d = data['mag_noisy_4d'] 
        deno_mag_4d = data['deno_mag_4d']
        deno_split_4d = data['deno_split_4d']
        deno_cmplx_4d = data['deno_cmplx_4d']
        mask_brain = data['mask_brain']
        
        # Average across echo dimension
        mag_orig = np.mean(mag_orig_4d, axis=3).astype(np.float32)
        mag_noisy = np.mean(mag_noisy_4d, axis=3).astype(np.float32)
        deno_mag = np.mean(deno_mag_4d, axis=3).astype(np.float32)
        deno_split = np.mean(deno_split_4d, axis=3).astype(np.float32)
        deno_cmplx = np.mean(deno_cmplx_4d, axis=3).astype(np.float32)
        
        # Transpose all data for axial view: (X,Y,Z) -> (Z,Y,X)
        print("🔄 Transposing axes for axial view...")
        mag_orig = np.transpose(mag_orig, (2, 1, 0))
        mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
        deno_mag = np.transpose(deno_mag, (2, 1, 0))
        deno_split = np.transpose(deno_split, (2, 1, 0))
        deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
        mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
        
        # Rotate images 90 degrees clockwise for better viewing
        print("🔄 Rotating images 90° clockwise...")
        mag_orig = np.rot90(mag_orig, k=-1, axes=(1, 2))      # k=-1 for clockwise
        mag_noisy = np.rot90(mag_noisy, k=-1, axes=(1, 2))
        deno_mag = np.rot90(deno_mag, k=-1, axes=(1, 2))
        deno_split = np.rot90(deno_split, k=-1, axes=(1, 2))
        deno_cmplx = np.rot90(deno_cmplx, k=-1, axes=(1, 2))
        mask_brain = np.rot90(mask_brain, k=-1, axes=(1, 2))
    
    print(f"\n📐 Final data shape: {mag_orig.shape}")
    print(f"📐 Mask shape: {mask_brain.shape}")
    print(f"📐 Axial slices: 0 to {mag_orig.shape[0]-1}")
    
    # Verify dimensions match
    if mag_orig.shape != mask_brain.shape:
        print("⚠️  Warning: Data and mask shapes don't match!")
        print(f"   Data shape: {mag_orig.shape}")
        print(f"   Mask shape: {mask_brain.shape}")
        return None
    
    # Create Napari viewer
    print("\n🎯 Creating Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - Axial View")
    
    # Set contrast limits
    print("📊 Computing contrast limits...")
    try:
        vmin, vmax = np.percentile(mag_orig[mask_brain.astype(bool)], [1, 99])
        print(f"📊 Contrast limits: {vmin:.1f} - {vmax:.1f}")
    except Exception as e:
        print(f"⚠️  Error computing contrast limits: {e}")
        print("   Using data min/max instead")
        vmin, vmax = np.percentile(mag_orig, [1, 99])
        print(f"📊 Fallback contrast limits: {vmin:.1f} - {vmax:.1f}")
    
    # Add main images
    viewer.add_image(mag_orig, name='1. Original', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy, name='2. Noisy', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag, name='3. Magnitude MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split, name='4. Split MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx, name='5. Complex MP-PCA', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add brain mask (hidden by default)
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    # Add difference maps (hidden by default)
    diff_vmax = np.percentile(np.abs(mag_noisy - mag_orig), 95)
    
    viewer.add_image((deno_mag - mag_orig), name='Diff: Mag - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_split - mag_orig), name='Diff: Split - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_cmplx - mag_orig), name='Diff: Complex - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    
    print("✅ Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Checkboxes: Show/hide layers") 
    print("  • Sliders: Adjust opacity")
    print("  • Enable 'Diff:' layers to see difference maps")
    
    return viewer

def load_4d_view():
    """Load 4D data for detailed echo-by-echo analysis"""
    
    print("📁 Loading 4D data from all_denoising_results.mat...")
    
    data = sio.loadmat('all_denoising_results.mat')
    
    # Load 4D data
    mag_orig_4d = data['mag_orig_4d'].astype(np.float32)
    mag_noisy_4d = data['mag_noisy_4d'].astype(np.float32)
    deno_mag_4d = data['deno_mag_4d'].astype(np.float32)
    deno_split_4d = data['deno_split_4d'].astype(np.float32)
    deno_cmplx_4d = data['deno_cmplx_4d'].astype(np.float32)
    mask_brain = data['mask_brain']
    
    # Transpose for axial view: (X,Y,Z,Ne) -> (Z,Y,X,Ne)
    print("🔄 Transposing 4D data for axial view...")
    mag_orig_4d = np.transpose(mag_orig_4d, (2, 1, 0, 3))
    mag_noisy_4d = np.transpose(mag_noisy_4d, (2, 1, 0, 3))
    deno_mag_4d = np.transpose(deno_mag_4d, (2, 1, 0, 3))
    deno_split_4d = np.transpose(deno_split_4d, (2, 1, 0, 3))
    deno_cmplx_4d = np.transpose(deno_cmplx_4d, (2, 1, 0, 3))
    mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
    
    # Rotate images 90 degrees clockwise for better viewing
    print("🔄 Rotating 4D images 90° clockwise...")
    mag_orig_4d = np.rot90(mag_orig_4d, k=-1, axes=(1, 2))      # k=-1 for clockwise
    mag_noisy_4d = np.rot90(mag_noisy_4d, k=-1, axes=(1, 2))
    deno_mag_4d = np.rot90(deno_mag_4d, k=-1, axes=(1, 2))
    deno_split_4d = np.rot90(deno_split_4d, k=-1, axes=(1, 2))
    deno_cmplx_4d = np.rot90(deno_cmplx_4d, k=-1, axes=(1, 2))
    mask_brain = np.rot90(mask_brain, k=-1, axes=(1, 2))
    
    print(f"📐 4D data shape: {mag_orig_4d.shape} (Z,Y,X,Echo)")
    print(f"📐 Axial slices: 0 to {mag_orig_4d.shape[0]-1}")
    print(f"📐 Echoes: 0 to {mag_orig_4d.shape[3]-1}")
    
    # Create Napari viewer
    print("\n🎯 Creating 4D Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - 4D View (All Echoes)")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig_4d[mask_brain.astype(bool)], [1, 99])
    
    # Add 4D images
    viewer.add_image(mag_orig_4d, name='1. Original (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_4d, name='2. Noisy (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_4d, name='3. Magnitude MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_4d, name='4. Split MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_4d, name='5. Complex MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    
    print("✅ 4D Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Shift + scroll: Navigate through echoes")
    
    return viewer

if __name__ == "__main__":
    import sys
    
    print("🚀 MAT Data Napari Viewer")
    print("=" * 30)
    
    if len(sys.argv) > 1 and sys.argv[1] == "4d":
        print("📊 Loading 4D view (all echoes)...")
        viewer = load_4d_view()
    else:
        print("📊 Loading 3D view (echo-averaged)...")
        viewer = load_and_view()
    
    if viewer is not None:
        napari.run()
    else:
        print("❌ Failed to create viewer")

IndentationError: unexpected indent (975982015.py, line 59)

In [13]:
import napari
import scipy.io as sio
import numpy as np

def load_and_view():
    """Load all_denoising_results.mat and view Echo 0 in Napari with correct axis orientation"""
    
    print("📁 Loading all_denoising_results.mat...")
    
    # Load the data
    data = sio.loadmat('all_denoising_results.mat')
    
    # Check what's available
    print("📊 Available data in MAT file:")
    for key in data.keys():
        if not key.startswith('__'):
            if isinstance(data[key], np.ndarray):
                print(f"  {key}: {data[key].shape}")
            else:
                print(f"  {key}: {type(data[key])}")
    
    # Load the 4D data to extract echo 0
    print("\n🔄 Extracting Echo 0 from 4D data...")
    mag_orig_4d = data['mag_orig_4d']
    mag_noisy_4d = data['mag_noisy_4d'] 
    deno_mag_4d = data['deno_mag_4d']
    deno_split_4d = data['deno_split_4d']
    deno_cmplx_4d = data['deno_cmplx_4d']
    mask_brain = data['mask_brain']
    
    # Extract echo 0 only
    mag_orig = mag_orig_4d[:, :, :, 0].astype(np.float32)
    mag_noisy = mag_noisy_4d[:, :, :, 0].astype(np.float32)
    deno_mag = deno_mag_4d[:, :, :, 0].astype(np.float32)
    deno_split = deno_split_4d[:, :, :, 0].astype(np.float32)
    deno_cmplx = deno_cmplx_4d[:, :, :, 0].astype(np.float32)
    
    print(f"📐 Echo 0 data shape: {mag_orig.shape}")
    
    # Transpose for axial view: (X,Y,Z) -> (Z,Y,X)
    print("🔄 Transposing axes for axial view...")
    mag_orig = np.transpose(mag_orig, (2, 1, 0))
    mag_noisy = np.transpose(mag_noisy, (2, 1, 0))
    deno_mag = np.transpose(deno_mag, (2, 1, 0))
    deno_split = np.transpose(deno_split, (2, 1, 0))
    deno_cmplx = np.transpose(deno_cmplx, (2, 1, 0))
    mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
    
    print(f"\n📐 Final data shape: {mag_orig.shape}")
    print(f"📐 Mask shape: {mask_brain.shape}")
    print(f"📐 Axial slices: 0 to {mag_orig.shape[0]-1}")
    print(f"📊 Showing Echo 0 data only")
    
    # Verify dimensions match
    if mag_orig.shape != mask_brain.shape:
        print("⚠️  Warning: Data and mask shapes don't match!")
        print(f"   Data shape: {mag_orig.shape}")
        print(f"   Mask shape: {mask_brain.shape}")
        return None
    
    # Create Napari viewer
    print("\n🎯 Creating Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - Echo 0 Only")
    
    # Set contrast limits
    print("📊 Computing contrast limits...")
    try:
        vmin, vmax = np.percentile(mag_orig[mask_brain.astype(bool)], [1, 99])
        print(f"📊 Contrast limits: {vmin:.1f} - {vmax:.1f}")
    except Exception as e:
        print(f"⚠️  Error computing contrast limits: {e}")
        print("   Using data min/max instead")
        vmin, vmax = np.percentile(mag_orig, [1, 99])
        print(f"📊 Fallback contrast limits: {vmin:.1f} - {vmax:.1f}")
    
    # Add main images
    viewer.add_image(mag_orig, name='1. Original (Echo 0)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy, name='2. Noisy (Echo 0)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag, name='3. Magnitude MP-PCA (Echo 0)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split, name='4. Split MP-PCA (Echo 0)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx, name='5. Complex MP-PCA (Echo 0)', colormap='gray', contrast_limits=[vmin, vmax])
    
    # Add brain mask (hidden by default)
    viewer.add_image(mask_brain, name='Brain Mask', colormap='red', opacity=0.3, visible=False)
    
    # Add difference maps (hidden by default)
    diff_vmax = np.percentile(np.abs(mag_noisy - mag_orig), 95)
    
    viewer.add_image((deno_mag - mag_orig), name='Diff: Mag - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_split - mag_orig), name='Diff: Split - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    viewer.add_image((deno_cmplx - mag_orig), name='Diff: Complex - Orig', 
                    colormap='bwr', contrast_limits=[-diff_vmax, diff_vmax], visible=False)
    
    print("✅ Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Checkboxes: Show/hide layers") 
    print("  • Sliders: Adjust opacity")
    print("  • Enable 'Diff:' layers to see difference maps")
    print("  • All data is from Echo 0 only")
    
    return viewer

def load_4d_view():
    """Load 4D data for detailed echo-by-echo analysis"""
    
    print("📁 Loading 4D data from all_denoising_results.mat...")
    
    data = sio.loadmat('all_denoising_results.mat')
    
    # Load 4D data
    mag_orig_4d = data['mag_orig_4d'].astype(np.float32)
    mag_noisy_4d = data['mag_noisy_4d'].astype(np.float32)
    deno_mag_4d = data['deno_mag_4d'].astype(np.float32)
    deno_split_4d = data['deno_split_4d'].astype(np.float32)
    deno_cmplx_4d = data['deno_cmplx_4d'].astype(np.float32)
    mask_brain = data['mask_brain']
    
    # Transpose for axial view: (X,Y,Z,Ne) -> (Z,Y,X,Ne)
    print("🔄 Transposing 4D data for axial view...")
    mag_orig_4d = np.transpose(mag_orig_4d, (2, 1, 0, 3))
    mag_noisy_4d = np.transpose(mag_noisy_4d, (2, 1, 0, 3))
    deno_mag_4d = np.transpose(deno_mag_4d, (2, 1, 0, 3))
    deno_split_4d = np.transpose(deno_split_4d, (2, 1, 0, 3))
    deno_cmplx_4d = np.transpose(deno_cmplx_4d, (2, 1, 0, 3))
    mask_brain = np.transpose(mask_brain, (2, 1, 0)).astype(np.float32)
    
    print(f"📐 4D data shape: {mag_orig_4d.shape} (Z,Y,X,Echo)")
    print(f"📐 Axial slices: 0 to {mag_orig_4d.shape[0]-1}")
    print(f"📐 Echoes: 0 to {mag_orig_4d.shape[3]-1}")
    
    # Create Napari viewer
    print("\n🎯 Creating 4D Napari viewer...")
    viewer = napari.Viewer(title="MP-PCA Denoising Results - 4D View (All Echoes)")
    
    # Set contrast limits
    vmin, vmax = np.percentile(mag_orig_4d[mask_brain.astype(bool)], [1, 99])
    
    # Add 4D images
    viewer.add_image(mag_orig_4d, name='1. Original (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(mag_noisy_4d, name='2. Noisy (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_mag_4d, name='3. Magnitude MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_split_4d, name='4. Split MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    viewer.add_image(deno_cmplx_4d, name='5. Complex MP-PCA (4D)', colormap='gray', contrast_limits=[vmin, vmax])
    
    print("✅ 4D Napari viewer ready!")
    print("\n🎮 Controls:")
    print("  • Mouse wheel: Navigate through axial slices")
    print("  • Shift + scroll: Navigate through echoes")
    
    return viewer

if __name__ == "__main__":
    import sys
    
    print("🚀 MAT Data Napari Viewer")
    print("=" * 30)
    
    if len(sys.argv) > 1 and sys.argv[1] == "4d":
        print("📊 Loading 4D view (all echoes)...")
        viewer = load_4d_view()
    else:
        print("📊 Loading Echo 0 view...")
        viewer = load_and_view()
    
    if viewer is not None:
        napari.run()
    else:
        print("❌ Failed to create viewer")

🚀 MAT Data Napari Viewer
📊 Loading Echo 0 view...
📁 Loading all_denoising_results.mat...
📊 Available data in MAT file:
  mag_orig_4d: (256, 224, 176, 6)
  mag_noisy_4d: (256, 224, 176, 6)
  deno_mag_4d: (256, 224, 176, 6)
  deno_split_4d: (256, 224, 176, 6)
  deno_cmplx_4d: (256, 224, 176, 6)
  mag_orig_avg: (176, 224, 256)
  mag_noisy_avg: (176, 224, 256)
  deno_mag_avg: (176, 224, 256)
  deno_split_avg: (176, 224, 256)
  deno_cmplx_avg: (176, 224, 256)
  mask_brain_transposed: (176, 224, 256)
  real_den_cmplx: (256, 224, 176, 6)
  imag_den_cmplx: (256, 224, 176, 6)
  den_real: (256, 224, 176, 6)
  den_imag: (256, 224, 176, 6)
  mask_brain: (256, 224, 176)
  data_info: (1,)
  dimensions: (1, 4)
  patch_radius: (1, 1)
  slice_idx: (1, 1)

🔄 Extracting Echo 0 from 4D data...
📐 Echo 0 data shape: (256, 224, 176)
🔄 Transposing axes for axial view...

📐 Final data shape: (176, 224, 256)
📐 Mask shape: (176, 224, 256)
📐 Axial slices: 0 to 175
📊 Showing Echo 0 data only

🎯 Creating Napari vie